# 🛰️ Generative Assembly — Scaling Study Notebook

## Before running this notebook

Run the scaling study script on your VM first:
```bash
chmod +x run_scaling_study.sh
./run_scaling_study.sh
```

This runs the geometry search at **9 candidate counts** (16 → 4096) on the **same 48 dev objects each time**, producing consistent data for a smooth scaling curve.

Expected output directories:
```
$GA_ROOT/runs/scaling-study-n16/evaluation/dev/metrics.jsonl
$GA_ROOT/runs/scaling-study-n32/evaluation/dev/metrics.jsonl
...
$GA_ROOT/runs/scaling-study-n4096/evaluation/dev/metrics.jsonl
```

---

## 🔧 Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "plotly", "pandas", "numpy", "open3d", "scipy"], check=True)
print("✅ Done")

In [ ]:
import os, json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from scipy.interpolate import make_interp_spline
import warnings
warnings.filterwarnings('ignore')
print("✅ Imports done")

## 📂 Discover Scaling Study Runs

In [ ]:
GA_ROOT    = os.environ.get("GA_ROOT",    "/home/kpandey/generative_assembly")
GA_DATA    = os.environ.get("GA_DATA",    "/home/kpandey/generative_assembly/bottles498-inputs-v1/dataset.json")
GA_CONFIG  = os.environ.get("GA_CONFIG",  "/home/kpandey/generative_assembly/configs/base.json")

SUCCESS_THRESHOLD = 0.01
GLOBAL_ARMS       = ["none__global", "true_shape__global", "wrong_shape__global", "distorted_shape__global"]
ARM_COLORS        = {
    "none__global":           "#e74c3c",
    "true_shape__global":     "#2ecc71",
    "wrong_shape__global":    "#3498db",
    "distorted_shape__global":"#f39c12",
}

# ── Auto-discover all scaling-study-n* runs ────────────────────────────────
RUNS_DIR = Path(GA_ROOT) / "runs"
runs = {}
for d in sorted(RUNS_DIR.glob("scaling-study-n*")):
    metrics_path = d / "evaluation/dev/metrics.jsonl"
    if metrics_path.exists():
        # Extract N from directory name
        n = int(d.name.split("-n")[-1])
        runs[n] = metrics_path

if not runs:
    print("❌ No scaling study runs found yet.")
    print(f"   Expected dirs like: {RUNS_DIR}/scaling-study-n32/evaluation/dev/metrics.jsonl")
    print("   Run ./run_scaling_study.sh on the VM first.")
else:
    print(f"✅ Found {len(runs)} completed runs:")
    for n, p in sorted(runs.items()):
        print(f"   N={n:5d}  →  {p}")

## 📥 Load and Aggregate Results

In [ ]:
def load_metrics(path: Path) -> pd.DataFrame:
    rows = []
    for line in open(path):
        row = json.loads(line)
        m = row.get("metrics") or {}
        rows.append({
            "case_id":              row.get("base_case_id", row.get("case_id")),
            "stage":                row.get("stage"),
            "arm":                  row.get("arm"),
            "best_candidate_error": m.get("best_candidate_error"),
            "max_part_chamfer":     m.get("max_part_chamfer"),
            "candidate_recall":     m.get("candidate_recall"),
            "success":              m.get("success"),
        })
    return pd.DataFrame(rows)

# Build a single long-form DataFrame: (n_candidates, arm, case_id, error)
records = []
for n_cands, path in sorted(runs.items()):
    df = load_metrics(path)
    e3 = df[(df["stage"] == "E3") & (df["arm"].isin(GLOBAL_ARMS))]
    for _, row in e3.iterrows():
        records.append({
            "n_candidates":         n_cands,
            "arm":                  row["arm"],
            "case_id":              row["case_id"],
            "best_candidate_error": row["best_candidate_error"],
            "max_part_chamfer":     row["max_part_chamfer"],
            "candidate_recall":     row["candidate_recall"],
            "success":              row["success"],
        })

long_df = pd.DataFrame(records)

if long_df.empty:
    print("⚠️  No data loaded. Run the scaling study first.")
else:
    print(f"Loaded {len(long_df)} rows")
    print(f"Candidate counts: {sorted(long_df['n_candidates'].unique())}")
    print(f"Cases per run:    {long_df.groupby('n_candidates')['case_id'].nunique().to_dict()}")
    print(f"Arms:             {long_df['arm'].unique().tolist()}")

---
# 📊 Plot 1 — Average FINAL Error vs # Candidates

> This plots `max_part_chamfer` — the actual error **after** template-based refinement.
> Now that candidates are generated using the template, the Oracle lines should drop significantly faster than `none__global`.

In [ ]:
if not long_df.empty:
    # Aggregate per (n_candidates, arm)
    agg = (
        long_df
        .dropna(subset=["max_part_chamfer"])
        .groupby(["n_candidates", "arm"])["max_part_chamfer"]
        .agg(
            mean_error="mean",
            std_error="std",
            median_error="median",
            p25=lambda x: x.quantile(0.25),
            p75=lambda x: x.quantile(0.75),
            n="count",
        )
        .reset_index()
    )

    fig = go.Figure()

    for arm in GLOBAL_ARMS:
        arm_data = agg[agg["arm"] == arm].sort_values("n_candidates")
        if arm_data.empty:
            continue
        clr = ARM_COLORS.get(arm, "#95a5a6")
        xs = arm_data["n_candidates"].values

        # Smooth the mean line using spline (only if enough points)
        if len(xs) >= 4:
            xs_smooth = np.logspace(np.log10(xs.min()), np.log10(xs.max()), 200)
            spline = make_interp_spline(np.log10(xs), arm_data["mean_error"].values, k=3)
            ys_smooth = np.clip(spline(np.log10(xs_smooth)), 0, None)
        else:
            xs_smooth = xs
            ys_smooth = arm_data["mean_error"].values

        # ── Shaded IQR band ─────────────────────────────────────────────────
        fig.add_trace(go.Scatter(
            x=np.concatenate([xs, xs[::-1]]),
            y=np.concatenate([arm_data["p75"].values, arm_data["p25"].values[::-1]]),
            fill="toself",
            fillcolor=clr.replace("#", "rgba(") + ",0.12)"
                if clr.startswith("#") else f"rgba(150,150,150,0.12)",
            line=dict(color="rgba(0,0,0,0)"),
            name=f"{arm} IQR",
            legendgroup=arm,
            showlegend=False,
            hoverinfo="skip",
        ))

        # ── Smooth mean line ─────────────────────────────────────────────────
        fig.add_trace(go.Scatter(
            x=xs_smooth, y=ys_smooth,
            mode="lines",
            name=arm,
            line=dict(color=clr, width=3),
            legendgroup=arm,
            showlegend=True,
        ))

        # ── Raw data points ──────────────────────────────────────────────────
        fig.add_trace(go.Scatter(
            x=xs, y=arm_data["mean_error"].values,
            mode="markers",
            name=arm,
            marker=dict(size=10, color=clr,
                        line=dict(color="white", width=1.5)),
            legendgroup=arm,
            showlegend=False,
            hovertemplate=(
                f"<b>{arm}</b><br>"
                "Candidates: %{x}<br>"
                "Mean final error: %{y:.4f}<br>"
                "N cases: %{customdata}<extra></extra>"
            ),
            customdata=arm_data["n"].values,
        ))

    # ── Success threshold ────────────────────────────────────────────────────
    fig.add_hline(
        y=SUCCESS_THRESHOLD,
        line_dash="dash", line_color="#ffffff", line_width=2,
        annotation_text=" ← Success threshold (0.01)",
        annotation_position="right",
        annotation_font=dict(color="white", size=12),
    )

    fig.update_xaxes(
        type="log",
        title="# Candidates (log scale)",
        tickvals=sorted(long_df["n_candidates"].unique()),
        ticktext=[str(v) for v in sorted(long_df["n_candidates"].unique())],
        showgrid=True, gridcolor="rgba(255,255,255,0.1)",
    )
    fig.update_yaxes(
        title="Average Final Error (max_part_chamfer)",
        showgrid=True, gridcolor="rgba(255,255,255,0.1)",
    )
    fig.update_layout(
        title=(
            "📉 Average Final Error (After Refinement) vs. # Candidates — 48 Objects<br>"
            "<sup>Shaded band = IQR across 48 objects | "
            "Smooth line = mean | "
            "Dots = actual run data points</sup>"
        ),
        template="plotly_dark",
        height=580,
        hovermode="x unified",
        legend=dict(orientation="h", y=-0.18),
    )
    fig.show()

    # Print summary table
    print("\n📋 Mean Final Error by Candidate Count:")
    pivot = agg.pivot_table(index="n_candidates", columns="arm",
                             values="mean_error").round(4)
    print(pivot.to_string())

---
# 📊 Plot 2 — Success Rate vs # Candidates

> What fraction of the 48 objects succeed at each candidate count?

In [ ]:
if not long_df.empty:
    recall_agg = (
        long_df
        .groupby(["n_candidates", "arm"])
        .agg(recall_rate=("candidate_recall", "mean"),
             success_rate=("success", "mean"),
             n=("case_id", "count"))
        .reset_index()
    )

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Candidate Recall Rate (%) — Did we generate a good guess?",
                        "Final Success Rate (%) — Did it successfully assemble?"]
    )

    for arm in GLOBAL_ARMS:
        arm_data = recall_agg[recall_agg["arm"] == arm].sort_values("n_candidates")
        if arm_data.empty: continue
        clr = ARM_COLORS.get(arm, "#95a5a6")

        fig.add_trace(go.Scatter(
            x=arm_data["n_candidates"],
            y=arm_data["recall_rate"] * 100,
            mode="lines+markers", name=arm,
            line=dict(color=clr, width=2.5),
            marker=dict(size=9),
            legendgroup=arm, showlegend=True,
            hovertemplate=f"<b>{arm}</b><br>N=%{{x}}<br>Recall=%{{y:.1f}}%<extra></extra>"
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=arm_data["n_candidates"],
            y=arm_data["success_rate"] * 100,
            mode="lines+markers", name=arm,
            line=dict(color=clr, width=2.5, dash="dot"),
            marker=dict(size=9),
            legendgroup=arm, showlegend=False,
            hovertemplate=f"<b>{arm}</b><br>N=%{{x}}<br>Success=%{{y:.1f}}%<extra></extra>"
        ), row=1, col=2)

    fig.update_xaxes(type="log", title="# Candidates")
    fig.update_yaxes(title="%", range=[0, 105])
    fig.update_layout(
        title="Recall & Success Rate vs. # Candidates — 48 Objects",
        template="plotly_dark", height=500,
        legend=dict(orientation="h", y=-0.2)
    )
    fig.show()

---
# 📊 Plot 3 — Error Distribution at Each Candidate Count (Violin)

> This shows HOW the distribution shifts as we add more candidates.
> Focus on `true_shape__global`.

In [ ]:
ARM_FOCUS = "true_shape__global"   # ← Change to any arm you want

if not long_df.empty:
    subset = long_df[
        (long_df["arm"] == ARM_FOCUS) &
        long_df["max_part_chamfer"].notna()
    ].copy()
    subset["n_str"] = subset["n_candidates"].astype(str)  # for categorical axis

    fig = go.Figure()
    cand_counts = sorted(subset["n_candidates"].unique())
    palette = px.colors.sequential.Viridis_r

    for i, n in enumerate(cand_counts):
        data = subset[subset["n_candidates"] == n]["max_part_chamfer"].values
        clr = palette[int(i / len(cand_counts) * (len(palette)-1))]

        fig.add_trace(go.Violin(
            x=[str(n)] * len(data),
            y=data,
            name=f"N={n}",
            line_color=clr,
            fillcolor=clr.replace("rgb", "rgba").replace(")", ",0.3)") if clr.startswith("rgb") else clr,
            box_visible=True,
            meanline_visible=True,
            points="all",
            pointpos=0,
            jitter=0.3,
            marker=dict(size=4, opacity=0.5),
            spanmode="hard",
        ))

    fig.add_hline(y=SUCCESS_THRESHOLD, line_dash="dash", line_color="white", line_width=2,
                  annotation_text=" Success threshold (0.01)",
                  annotation_font=dict(color="white"))

    fig.update_yaxes(title="Final Error (max_part_chamfer)")
    fig.update_xaxes(title="# Candidates",
                     categoryorder="array",
                     categoryarray=[str(n) for n in cand_counts])
    fig.update_layout(
        title=f"Final Error Distribution as Candidates Increase — Arm: {ARM_FOCUS}<br>"
              "<sup>Each violin = 48 objects. Box = median/IQR. Dots = individual cases.</sup>",
        template="plotly_dark", height=600, showlegend=False
    )
    fig.show()

---
# 📊 Plot 4 — Success Rate vs Relaxed Threshold

> Using the **highest candidate count run** (most data).

In [ ]:
if not long_df.empty:
    # Use the run with the most candidates
    max_n = long_df["n_candidates"].max()
    best_run = long_df[long_df["n_candidates"] == max_n]

    thresholds = np.unique(np.concatenate([
        np.linspace(0.001, 0.01, 25),
        np.linspace(0.01,  0.05, 25),
        np.linspace(0.05,  0.15, 10),
    ]))

    fig = go.Figure()

    for arm in GLOBAL_ARMS:
        subset = best_run[
            (best_run["arm"] == arm) &
            best_run["max_part_chamfer"].notna()
        ]["max_part_chamfer"].values
        if len(subset) == 0: continue

        success_rates = [(subset < t).mean() * 100 for t in thresholds]
        clr = ARM_COLORS.get(arm, "#95a5a6")

        fig.add_trace(go.Scatter(
            x=thresholds, y=success_rates,
            mode="lines", name=arm,
            line=dict(color=clr, width=2.5),
            fill="tozeroy" if arm == "true_shape__global" else None,
            fillcolor=clr.replace("#","rgba(") + ",0.07)" if clr.startswith("#") else None,
            hovertemplate="Threshold: %{x:.3f}<br>Success: %{y:.1f}%<extra></extra>"
        ))

    fig.add_vline(x=0.01, line_dash="dash", line_color="#e74c3c", line_width=2,
                  annotation_text=" Current (0.01)", annotation_font_color="#e74c3c")
    fig.add_vline(x=0.03, line_dash="dot", line_color="#f39c12", line_width=1.5,
                  annotation_text=" Relaxed (0.03)", annotation_font_color="#f39c12")

    fig.update_xaxes(title="Final Error Threshold (lower = stricter)")
    fig.update_yaxes(title="% of 48 Cases that Succeed", range=[0, 105])
    fig.update_layout(
        title=f"📈 Success Rate vs. Threshold — {max_n} Candidates, 48 Objects<br>"
              "<sup>If the curve jumps sharply right after 0.01, we are barely failing. If it's gradual, errors are large.</sup>",
        template="plotly_dark", height=520,
        legend=dict(orientation="h", y=-0.2)
    )
    fig.show()

    # Table at key thresholds
    key_t = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10]
    rows = []
    for arm in GLOBAL_ARMS:
        subset = best_run[
            (best_run["arm"] == arm) &
            best_run["max_part_chamfer"].notna()
        ]["max_part_chamfer"].values
        for t in key_t:
            rows.append({"Arm": arm, "Threshold": t,
                         "Success": f"{(subset < t).mean()*100:.1f}%",
                         "N": f"{(subset < t).sum()}/{len(subset)}"})
    tbl = pd.DataFrame(rows).pivot_table(index="Threshold", columns="Arm",
                                          values="Success", aggfunc="first")
    print(f"\n📋 Success rates at key thresholds (N={max_n} candidates):")
    print(tbl.to_string())

---
# 🧊 Ground Truth Bottle Visualization

Loads actual fragment point clouds from `dataset.json`'s `.npz` files.

In [ ]:
import open3d as o3d

dataset_path = Path(GA_DATA)
dataset = json.loads(dataset_path.read_text()) if dataset_path.exists() else None
DATASET_DIR = dataset_path.parent

if dataset:
    print(f"✅ Loaded dataset.json")
    cases = dataset.get("cases", dataset) if isinstance(dataset, dict) else dataset
else:
    print("❌ dataset.json not found")

In [ ]:
EXAMPLE_IDX = 0  # ← Change to see different objects

def load_gt_case(case_record):
    npz_path = DATASET_DIR / case_record["path"]
    if not npz_path.exists():
        return None
    with np.load(npz_path, allow_pickle=False) as f:
        frag_keys = sorted([k for k in list(f.files) if k.startswith("points_")])
        return [f[k] for k in frag_keys]

if dataset:
    entry = cases[EXAMPLE_IDX]
    print(f"Case ID: {entry.get('id', '?')}")
    fragments_gt = load_gt_case(entry)
    if fragments_gt:
        print(f"Loaded {len(fragments_gt)} fragments from {entry['path']}")
    else:
        print(f"❌ Could not load {entry['path']}")

In [ ]:
# ── Visualize GT assembly ──────────────────────────────────────────────────
PALETTE = px.colors.qualitative.Plotly
fig = go.Figure()

if dataset and 'fragments_gt' in locals() and fragments_gt:
    for i, pts in enumerate(fragments_gt):
        fig.add_trace(go.Scatter3d(
            x=pts[:,0], y=pts[:,1], z=pts[:,2],
            mode="markers",
            marker=dict(size=2, color=PALETTE[i % len(PALETTE)], opacity=0.85),
            name=f"Fragment {i+1}"
        ))

fig.update_layout(
    title="🍾 Ground Truth Bottle Assembly — Each color = one fragment",
    scene=dict(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
        bgcolor="#0d1117",
        aspectmode="data",   # preserves real proportions
    ),
    template="plotly_dark", height=680,
)
fig.show()

---
# 🔍 Key Takeaways

| Finding | What to look for in plots above |
|---|---|
| Does the Oracle shape prior work now? | Plot 1: Is `true_shape__global` significantly lower than `none__global`? |
| Did we hit a ceiling? | Plot 1: Flat curve = ceiling hit. |
| Is threshold the issue? | Plot 4: Sharp jump after 0.01 = threshold too strict. Gradual = real errors. |